In [2]:
from datamodel import TradingState, Listing, OrderDepth, Observation, Symbol, Trade, Position, Order
from backtester import Backtester
from typing import Dict, List
from trader import Trader
import matplotlib.pyplot as plt
import numpy as np

In [3]:
class UserTrader(Trader):

    def __init__(self):
        # Keep track of previous mid-price for Tomatoes
        self.prev_tomato_mid = None

    def bid(self):
        return 15
    
    def run(self, state: TradingState):
        result = {}
        
        for product, order_depth in state.order_depths.items():
            orders: List[Order] = []

            # --- EMERALDS: simple MM around 10000 ---
            if product == "EMERALDS":
                FAIR_PRICE = 10000
                SPREAD = 2
                POSITION_LIMIT = 20
                position = state.position.get(product, 0)
                base_size = 10

                best_bid = max(order_depth.buy_orders.keys(), default=0)
                best_ask = min(order_depth.sell_orders.keys(), default=999999)

                # Place buy order slightly below fair price
                if position < POSITION_LIMIT:
                    buy_qty = min(base_size, POSITION_LIMIT - position)
                    orders.append(Order(product, FAIR_PRICE - SPREAD, buy_qty))
                
                # Place sell order slightly above fair price
                if position > -POSITION_LIMIT:
                    sell_qty = min(base_size, POSITION_LIMIT + position)
                    orders.append(Order(product, FAIR_PRICE + SPREAD, -sell_qty))

            # --- TOMATOES: simple momentum (trend) strategy ---
            elif product == "TOMATOES":
                POSITION_LIMIT = 20
                position = state.position.get(product, 0)
                base_size = 10

                best_bid = max(order_depth.buy_orders.keys(), default=0)
                best_ask = min(order_depth.sell_orders.keys(), default=999999)

                if best_bid == 0 or best_ask == 999999:
                    result[product] = orders
                    continue  # Skip if no market

                mid_price = (best_bid + best_ask) / 2

                # Uptrend → buy, Downtrend → sell
                if self.prev_tomato_mid is not None:
                    if mid_price > self.prev_tomato_mid and position < POSITION_LIMIT:
                        buy_qty = min(base_size, POSITION_LIMIT - position)
                        orders.append(Order(product, best_ask, buy_qty))
                    elif mid_price < self.prev_tomato_mid and position > -POSITION_LIMIT:
                        sell_qty = min(base_size, POSITION_LIMIT + position)
                        orders.append(Order(product, best_bid, -sell_qty))

                self.prev_tomato_mid = mid_price

            result[product] = orders

        # Trader state string (can store any info)
        traderData = "SIMPLE_MM_TREND"

        # No conversions in this basic strategy
        conversions = 0

        return result, conversions, traderData

In [4]:
backtesting_engine = Backtester("./data/prices.csv", "./data/trades.csv")

c:\Projects\IMC_PROSPERITY_4\backtesting_engine\data_parser.py:12: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace("", 0)
c:\Projects\IMC_PROSPERITY_4\backtesting_engine\data_parser.py:16: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["buyer"].replace(0, "", inplace=True)
c:\Project

In [5]:
sample_trader = UserTrader()
pos, pnl=backtesting_engine.run_trader(sample_trader)

In [10]:
uniq = {}
for d in pnl:
    if d["TOMATOES"] not in uniq:
        uniq[d["TOMATOES"]]=True

In [13]:
uniq

{0.0: True,
 -40156.0: True,
 -85341.0: True,
 -100354.0: True,
 -250474.0: True,
 -370531.0: True,
 -405677.0: True,
 -430730.0: True,
 -520814.0: True,
 -565994.5: True,
 -701066.0: True,
 -736153.0: True,
 -826262.5: True,
 -901302.0: True,
 -1021356.0: True,
 -1096374.0: True,
 -1246404.5: True,
 -1261374.0: True,
 -1261364.0: True,
 -1261354.0: True,
 -1261344.0: True,
 -1311529.0: True,
 -1336624.0: True,
 -1426589.0: True,
 -1516550.5: True,
 -1561712.0: True,
 -1711653.0: True,
 -1741618.0: True,
 -1741608.0: True,
 -1786772.0: True,
 -1821886.0: True,
 -1911801.0: True,
 -1936903.5: True,
 -1982050.0: True,
 -2027187.5: True,
 -2062226.0: True,
 -2167256.5: True,
 -2197346.0: True,
 -2202352.0: True,
 -2337290.0: True,
 -2377408.5: True,
 -2382406.0: True,
 -2502376.0: True,
 -2542502.0: True,
 -2542482.0: True,
 -2632480.0: True,
 -2662578.0: True,
 -2662568.0: True,
 -2797582.5: True,
 -2837719.5: True,
 -2842730.0: True,
 -2842710.0: True,
 -2842700.0: True,
 -2992780.0: Tr